# pane

> What the main agent and each sub-agent are doing now, written to a file and drawn in a tmux split beside the chat.

In [ ]:
#| default_exp pane

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
from fastcore.test import test_eq

In [ ]:
#| export
from __future__ import annotations

import io, json, os, re, select, signal, textwrap, time, weakref
from pathlib import Path

from fastcore.basics import first
from fastcore.script import call_parse
from rich.console import Console, Group
from rich.text import Text
from teleprint.keys import Key, Mouse, Parser
from teleprint.tty import RealTty

from shalya.core import clip, diff_text, one_line as _1

from ramabana.monitor import _counts, _rel

## The snapshot

`now_snapshot` returns one plain dict, so the CLI can write it and another process can draw it.

- `root` is the foreground turn: its step count, run time, state, the call it waits on (`current`), and its `status`, the line its model last wrote before a call.
- `plan` is the agent's todo list, each item's text and status.
- `subs` holds one row per sub-agent: each delegation the turn started, and each background delegation running or finished since the turn began. A row carries its status, its last `CALLS_KEPT` calls with clipped results, and its clipped answer once it has finished. A finished sub-agent keeps its row until the next turn.
- `files` lists this turn's writes: path, lines added and removed, and the diff cut to `DIFF_LINES`.
- `notes` holds what the model said before each of this turn's calls, whole and timestamped, up to `NOTE_KEEP`. The status line is the clipped last line of the newest.
- `background` lists background delegations, `run_shell_bg` commands and folder watches by kind, id, label, state and seconds running. A finished shell or a watch has no clock, so its `elapsed` is `None`.

The CLI writes a snapshot up to five times a second, so one must be cheap when nothing changed. `files` is recomputed only after a call finishes. A shell's state is asked at most every `SHELL_EVERY` seconds while it runs, and never again once it has ended.

Every `Act` names the run whose model made it, so a sub-agent's row counts the calls stamped with its run id. A root step is counted as `Ui.working` counts it: a call whose `parent_action_id` names no call on the activity. A sub-agent's calls name the delegate call that started it, so they are never root steps.

In [ ]:
#| export
CALLS_KEPT = 8       #: a sub-agent's latest calls in a snapshot
OUT_CHARS = 200      #: chars kept of each call's result
ANSWER_CHARS = 500   #: chars kept of a finished sub-agent's answer
DIFF_LINES = 80      #: lines kept of each file's diff
SHELL_EVERY = 1.0    #: seconds between asks about a live background shell

class _Seen:
    "What earlier snapshots of one agent computed that is dear to compute again: its files, and each shell's state."
    def __init__(self): self.key, self.files, self.shells = None, [], {}

_SEEN = weakref.WeakKeyDictionary()

def _secs(r, now): return round(((r.ended or now) - r.started) if r.started else 0, 1)

def _current(acts): return next((a.summary for a in reversed(acts) if not a.done), '')

def _bg_runs(agent, since):
    "Background delegations that are live or finished since `since`."
    return [r for r in agent._side_runs() if r.kind == 'background' and (not r.terminal or r.ended >= since)]

def _sub(r, every, now):
    calls = [a for a in every if a.run_id == r.id]
    return {'id': r.id, 'question': _1(r.question, 200), 'elapsed': _secs(r, now), 'state': r.state, 'steps': len(calls),
            'status': r.status, 'current': _current(calls),
            'calls': [{'line': a.summary, 'ok': a.ok, 'done': a.done, 'out': clip(a.detail, OUT_CHARS)} for a in calls[-CALLS_KEPT:]],
            'answer': clip(r.answer, ANSWER_CHARS) if r.terminal else ''}

def _files(agent, seen, turn):
    "This turn's writes: path under the first root, lines added and removed, and the diff cut to `DIFF_LINES`. Recomputed only after a call finishes."
    key = (agent.current_turn_id, len(agent.before), sum(a.done for a in turn), turn[-1].id if turn else '')
    if key == seen.key: return seen.files
    root, out = first(agent.host.roots or ()), []
    for p, (was, now) in agent.changes().items():
        d = diff_text(was, now, rel := _rel(p, root))
        (added, removed), ls = _counts(d), d.splitlines()
        more = [f'… {len(ls) - DIFF_LINES} more lines'] if len(ls) > DIFF_LINES else []
        out.append({'path': rel, 'added': added, 'removed': removed, 'diff': '\n'.join(ls[:DIFF_LINES] + more)})
    seen.key, seen.files = key, out
    return out

_STARTED = re.compile(r'started (cmd_\w+)')

def _shell_state(agent, seen, rid, now):
    "`rid`'s state, asked of the host at most every `SHELL_EVERY` while it runs and never again once it has ended; '' when unknown."
    state, at = seen.shells.get(rid, (None, 0))
    if state is not None and (state != 'running' or now - at < SHELL_EVERY): return state
    try: state = agent.host.cmd_output(rid, 1)[0]
    except Exception: state = ''   # a host that runs no background commands, or has forgotten this one
    seen.shells[rid] = state, now
    return state

def _shells(agent, seen, every, since, now):
    "`run_shell_bg` commands the host still knows: live ones, and those started since `since`."
    out = []
    for a in every:
        if a.tool != 'run_shell_bg' or not (m := _STARTED.match(a.detail)): continue
        if a.started < since and seen.shells.get(m[1], ('running',))[0] != 'running': continue
        if (state := _shell_state(agent, seen, m[1], now)) and (state == 'running' or a.started >= since):
            out.append({'kind': 'shell', 'id': m[1], 'label': _1(a.args.get('command'), 120), 'state': state,
                        'elapsed': round(now - a.started, 1) if state == 'running' else None})
    return out

def _background(agent, seen, bg, every, since, now):
    "Background delegations, shells and folder watches, one row each."
    return ([{'kind': 'delegate', 'id': r.id, 'label': _1(r.question, 120), 'state': r.state, 'elapsed': _secs(r, now)} for r in bg]
            + _shells(agent, seen, every, since, now)
            + [{'kind': 'watch', 'id': w.id, 'label': w.folder, 'state': w.last_status or 'watching', 'elapsed': None} for w in agent.monitors.all()])

def now_snapshot(agent):
    "The foreground turn, its plan, sub-agents, writes and background work, as a JSON-able dict."
    (every, mark), now, root = agent.activity.window(), time.time(), agent.run()
    ids, since = {a.id for a in every}, root.started if root else 0
    mine, bg = [a for a in every[mark:] if a.parent_action_id not in ids], _bg_runs(agent, since)
    kids, seen = [c for c in (root.children if root else []) if c.kind == 'child'], _SEEN.setdefault(agent, _Seen())
    return {'at': now, 'busy': agent.busy,
            'root': {'turn_elapsed': _secs(root, now) if root else 0, 'steps': len(mine), 'state': root.state if root else 'idle',
                     'status': agent.status_line, 'current': _current(mine)},
            'plan': [{'text': t.text, 'status': t.status} for t in agent.plan.todos],
            'subs': [_sub(r, every, now) for r in kids + bg],
            'files': _files(agent, seen, every[mark:]),
            'background': _background(agent, seen, bg, every, since, now),
            'notes': [{'at': t, 'text': s} for t, s in (root.notes if root else [])]}

A turn that delegates: the scripted model reads a file, then asks one question. Its sub-agent reads the file too. The activity's `on_change` hook takes a snapshot as that read starts, when the pane shows both working.

In [ ]:
from ramabana.runtime import current_run
from ramabana.testing import ScriptedBackend, Step, fake_agent

READ = ('view_file', {'path': '/proj/a.py'})
a, _ = fake_agent()
seen = []
a.activity.on_change = lambda act: seen.append(now_snapshot(a)) if not act.done and current_run().kind != 'root' else None
class Root(ScriptedBackend):
    def spawn(self, sp='', tools=(), **kw):
        return ScriptedBackend(self.spec, steps=[Step(tool=READ), Step('a() is defined')], token_delay=0, tools=tools)
be = Root(steps=[Step(tool=READ), Step(tool=('delegate_search', {'questions': ['what does a.py define?']})), Step('done')],
          token_delay=0, tools=a.tools, sp=a.system_prompt())
a._be = a._be_or_none = lambda job='turn': be
list(a.stream('go'))
mid = seen[0]
test_eq((mid['busy'], mid['root']['steps'], [(s['state'], s['steps']) for s in mid['subs']]), (True, 2, [('running', 1)]))
mid

After the turn ends the sub-agent stays listed as finished. The next turn clears it.

In [ ]:
test_eq([s['state'] for s in now_snapshot(a)['subs']], ['completed'])
be.steps = [Step('ok')]
list(a.stream('again'))
test_eq(now_snapshot(a)['subs'], [])

The plan and this turn's writes ride along, each file with its diff.

In [ ]:
a.plan.set('ship', ['read a.py', 'fix it']); a.plan.update(a.plan.todos[0].id, status='active')
a.before['/proj/a.py'] = 'def a(): pass\n'; a.host.files['/proj/a.py'] = 'def a(): return 1\n'
snap = now_snapshot(a)
test_eq(snap['plan'], [{'text': 'read a.py', 'status': 'active'}, {'text': 'fix it', 'status': 'pending'}])
test_eq([(f['path'], f['added'], f['removed']) for f in snap['files']], [('a.py', 1, 1)])
print(snap['files'][0]['diff'])

`write_snapshot` replaces the file in one step, so a reader never sees half of it. `read_snapshot` returns `None` for a file that is missing or not yet JSON.

In [ ]:
#| export
def write_snapshot(agent, path):
    "Write `now_snapshot(agent)` to `path` atomically."
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_name(f'.{p.name}.{os.getpid()}.tmp')
    tmp.write_text(json.dumps(now_snapshot(agent)))
    os.replace(tmp, p)

def read_snapshot(path):
    "The snapshot at `path`, or `None` when it is missing or unreadable."
    try: return json.loads(Path(path).read_text())
    except (OSError, ValueError): return None

In [ ]:
import tempfile
with tempfile.TemporaryDirectory() as d:
    p = Path(d)/'now.json'
    test_eq(read_snapshot(p), None)
    write_snapshot(a, p)
    test_eq((read_snapshot(p)['busy'], [f.name for f in Path(d).iterdir()]), (False, ['now.json']))
    p.write_text('{"busy": tr')
    test_eq(read_snapshot(p), None)

## Drawing it

`board` lays a snapshot out as rows for a pane `width` cells wide, one row per screen line. Lines are cut to the width, not wrapped. From the top:

- a header: `main`, its step, its clock and its status line. The chat already shows the call the main agent waits on, so the board leaves it out. An idle agent gets one line.
- the plan as a checklist, with the active item highlighted.
- the sub-agents, one row each: its id, its status line (or its question when it has none), its clock and a mark (`▶` running, `✓` completed, `✗` stopped or failed). An open row (`▾`) adds the question, each recent call with its clipped result, and the answer once there is one.
- the files this turn wrote, each with lines added and removed. An open row shows the diff, `+` lines green and `-` lines red.
- the background work: kind, label, state and clock.
- the notes, newest first, for reading back what the model said along the way: each a `▸` row with its time and first line, opening (`▾`) to every word wrapped to the pane. The newest starts open.

Empty sections are left out. A row that opens carries a key, `('sub', id)`, `('file', path)` or `('note', at)`, so it stays open when the next snapshot arrives. A running clock advances by the time since the snapshot was written, so a long call does not look frozen between writes. The colours are the CLI's default `github-dark` palette. This module does not import `ramabana.cli`, because `ramabana.cli` imports it.

In [ ]:
#| export
PALETTE = {'fg0': '#e6edf3', 'fg1': '#c9d1d9', 'gray': '#8b949e', 'red': '#ff7b72', 'green': '#3fb950',
           'yellow': '#d29922', 'blue': '#58a6ff', 'cursor': '#30363d'}
_STOPPED = ('cancelled', 'detached', 'terminated', 'failed', 'stopped')
_TODO = {'done': ('✓', 'green', 'gray'), 'active': ('◐', 'yellow', 'bold yellow'), 'cancelled': ('✗', 'gray', 'strike gray')}

def _c(name): return ' '.join(PALETTE.get(w, w) for w in name.split())

def _mark(state):
    if state in ('completed', 'exit 0'): return '✓', PALETTE['green']
    return ('✗', PALETTE['red']) if state in _STOPPED or state.startswith('exit ') else ('▶', PALETTE['yellow'])

def _line(s, style, width, indent=''):
    t = Text(indent + s.expandtabs(4), style=style)
    t.truncate(width, overflow='ellipsis')
    return t

def _row(width, head, mid='', tail=None, style='', left=False):
    "`head`, `mid` and `tail` on one row `width` wide; `mid` gives way, from the `left` when a path."
    room = max(1, width - head.cell_len - (tail.cell_len if tail else 0))
    if left and len(mid) > room: mid = '…' + mid[len(mid) - room + 1:]
    t = head.copy()
    t.append_text(_line(mid, style, room))
    if tail: t.append_text(tail)
    t.truncate(width, overflow='ellipsis')
    return t

def _clock(secs, state, drift): return f'{secs + (drift if state == "running" else 0):.0f}s'
def _steps(n): return f'{n} step' + ('' if n == 1 else 's')

def _head(snap, width, drift):
    root, t = snap['root'], Text('main', style=f"bold {PALETTE['blue']}")
    if not snap['busy']: return _row(width, t, f" · idle · {_steps(root['steps'])}", style=PALETTE['gray'])
    step = f"step {root['steps']}" if root['steps'] else 'thinking'
    t.append(f" · {step} · {_clock(root['turn_elapsed'], root['state'], drift)}", style=PALETTE['gray'])
    return _row(width, t, f" — {_1(root['status'], 200)}" if root['status'] else '', style=PALETTE['fg0'])

def _todo(t, width):
    mark, ms, ts = _TODO.get(t['status'], ('○', 'gray', 'fg1'))
    return _row(width, Text(mark + ' ', style=_c(ms)), _1(t['text'], 200), style=_c(ts))

def _sub_rows(s, width, drift, opened):
    mark, style = _mark(s['state'])
    tail = Text(f" · {_clock(s['elapsed'], s['state'], drift)} ", style=PALETTE['gray']) + Text(mark, style=style)
    head = Text(('▾ ' if opened else '▸ ') + s['id'], style=f'bold {style}')
    out = [(('sub', s['id']), _row(width, head, ' · ' + _1(s['status'] or s['question'], 200), tail, PALETTE['fg1']))]
    if not opened: return out
    if s['status']: out.append((None, _line(_1(s['question'], 200), PALETTE['gray'], width, '  ')))
    for c in s['calls']:
        m, st = ('✓', 'green') if c['done'] and c['ok'] else ('✗', 'red') if c['done'] else ('⏳', 'yellow')
        out.append((None, _row(width, Text(f'  {m} ', style=PALETTE[st]), _1(c['line'], 200), style=PALETTE['fg1'])))
        if c['out']: out.append((None, _line(_1(c['out'], 200), PALETTE['gray'], width, '    ')))
    words = textwrap.wrap(_1(s['answer'], ANSWER_CHARS), max(8, width - 4))
    return out + [(None, _line(l, PALETTE['fg0'], width, '  → ' if i == 0 else '    ')) for i, l in enumerate(words)]

def _diff_style(l):
    if l.startswith(('+++', '---')): return PALETTE['gray']
    return PALETTE[{'+': 'green', '-': 'red', '@': 'blue'}.get(l[:1], 'fg1')]

def _file_rows(f, width, opened):
    tail = Text(f" +{f['added']}", style=PALETTE['green']) + Text(f" −{f['removed']}", style=PALETTE['red'])
    out = [(('file', f['path']), _row(width, Text('▾ ' if opened else '▸ ', style=PALETTE['gray']), f['path'], tail, PALETTE['fg0'], left=True))]
    return out + [(None, _line(l, _diff_style(l), width, '  ')) for l in f['diff'].splitlines()] if opened else out

def _bg_row(b, width, drift):
    mark, style = _mark(b['state'])
    clock = f" · {_clock(b['elapsed'], b['state'], drift)}" if b['elapsed'] is not None else ''
    head = Text(f'{mark} ', style=style) + Text(f"{b['kind']} ", style=PALETTE['gray'])
    return _row(width, head, _1(b['label'], 200), Text(f" · {b['state']}{clock}", style=PALETTE['gray']), PALETTE['fg1'])

def _note_rows(n, width, opened):
    "One note: `▸`, its time and first line; opened, `▾` and every word wrapped to the pane."
    head = Text(('▾ ' if opened else '▸ ') + time.strftime('%H:%M ', time.localtime(n['at'])), style=PALETTE['gray'])
    room, key = max(8, width - head.cell_len), ('note', n['at'])
    if not opened: return [(key, head + _line(n['text'].splitlines()[0], PALETTE['fg1'], room))]
    lines = [w for l in n['text'].splitlines() for w in textwrap.wrap(l, room) or ['']]
    return [(key if i == 0 else None, (head if i == 0 else Text(' ' * head.cell_len)) + _line(l, PALETTE['fg1'], room))
            for i, l in enumerate(lines)]

def board(snap, width=40, now=None, open=(), err=''):
    "The pane for `snap` as `(key, row)` pairs, one per screen line; a row with a `key` opens and closes."
    warn = [(None, _line('⚠ ' + err, PALETTE['red'], width))] if err else []
    if not snap: return [(None, _line('waiting for the agent…', PALETTE['gray'], width))] + warn
    drift = max(0, (now or time.time()) - snap['at'])
    rows = [(None, _head(snap, width, drift))] + warn
    def section(title, body):
        if body: rows.extend([(None, Text('')), (None, Text(title, style=f"bold {PALETTE['gray']}")), *body])
    section('Plan', [(None, _todo(t, width)) for t in snap['plan']])
    section('Sub-agents', [r for s in snap['subs'] for r in _sub_rows(s, width, drift, ('sub', s['id']) in open)])
    section('Files', [r for f in snap['files'] for r in _file_rows(f, width, ('file', f['path']) in open)])
    section('Background', [(None, _bg_row(b, width, drift)) for b in snap['background']])
    notes = snap.get('notes') or []   # newest first, and the newest starts open: toggling flips that
    section('Notes', [r for i, n in enumerate(reversed(notes)) for r in _note_rows(n, width, (('note', n['at']) in open) != (i == 0))])
    return rows

def render(snap, width=40, now=None, open=()):
    "The pane for `snap` as a rich renderable, `width` cells wide, with the rows keyed in `open` opened."
    return Group(*(t for _, t in board(snap, width, now, open)))

A busy snapshot written five seconds ago. Every row is closed:

In [ ]:
call = lambda line, ok=True, done=True, out='': {'line': line, 'ok': ok, 'done': done, 'out': out}
now = time.time()
busy = {'at': now - 5, 'busy': True,
        'root': {'turn_elapsed': 42, 'steps': 7, 'state': 'running', 'status': 'Checking the pane tests', 'current': 'Run shell: pytest'},
        'plan': [{'text': 'Read the viewer', 'status': 'done'}, {'text': 'Rewrite render', 'status': 'active'},
                 {'text': 'Wire the CLI', 'status': 'pending'}, {'text': 'Poll the file', 'status': 'cancelled'}],
        'subs': [{'id': 'run_aaa', 'question': 'which files import fastllm?', 'elapsed': 12, 'state': 'running', 'steps': 2,
                  'status': 'Searching for imports', 'current': 'Search code: fastllm',
                  'calls': [call('Search code: import fastllm', out='agent.py:12: import fastllm'), call('Search code: fastllm', None, False)],
                  'answer': ''},
                 {'id': 'run_bbb', 'question': 'is the pane tested?', 'elapsed': 8, 'state': 'completed', 'steps': 1, 'status': '',
                  'current': '', 'calls': [call('Read tests/test_pane.py')], 'answer': 'Yes, in tests/test_pane.py.'}],
        'files': [{'path': 'ramabana/pane.py', 'added': 1, 'removed': 1, 'diff': '@@ -1 +1 @@\n-from rich.live import Live\n+from teleprint.keys import Parser'}],
        'background': [{'kind': 'shell', 'id': 'cmd_1234abcd', 'label': 'pytest -x tests', 'state': 'running', 'elapsed': 30}]}
def show(snap, open=()):
    con = Console(record=True, width=40, color_system=None)
    con.print(render(snap, 40, now=now, open=open))
    return con.export_text()
out = show(busy)
assert '47s' in out and 'Run shell' not in out and '▸ run_aaa' in out and 'import fastllm' not in out
assert all(len(l) <= 40 for l in out.splitlines())
print(out)

Opening the running sub-agent and the file shows its calls and the diff:

In [ ]:
out = show(busy, {('sub', 'run_aaa'), ('file', 'ramabana/pane.py')})
assert '▾ run_aaa' in out and '✓ Search code: import fastllm' in out and '+from teleprint.keys import Parser' in out
print(out)

## The viewer

`Viewer` draws the board on a terminal: anything with `read(timeout)`, `write`, `size` and `restore`, as teleprint's `RealTty` has. `PaneTty` is that `RealTty` with a `read` that returns `None` at the end of input. The viewer switches to the alternate screen and turns on SGR mouse reports. Each `tick` reads input for up to `every` seconds and acts on it: a click opens or closes the row under it, and the wheel scrolls. `j`/`k` or the arrows move the cursor over the rows that open, and scroll on past the first and last of them, so a board taller than the pane can be read to its end. Page up and page down scroll a page, enter or space toggle the cursor's row, `g` and `G` jump to the top and the bottom, and `q` or ctrl+c quit.

The snapshot is read again only when its file changes. The board is laid out every tick, but only the lines that changed are written, so a clock that moves once a second rewrites one line a second and a still board writes nothing. A new terminal size clears and redraws everything.

The viewer must outlive anything a frame throws. An error keeps the last good snapshot and shows itself as a row until the next good one arrives. On the way out, however it leaves, the terminal gets its screen and modes back. A quit leaves `quit_mark` beside the snapshot, `now.closed`, so the CLI tells a viewer the user closed from one that crashed. A hangup (tmux closing the pane), SIGTERM and the end of input are quits too.

In [ ]:
#| export
ENTER = '\x1b[?1049h\x1b[?25l\x1b[?1000;1006h'   #: alt screen, cursor hidden, SGR mouse clicks on
LEAVE = '\x1b[?1000;1006l\x1b[?25h\x1b[?1049l'
WHEEL = 3   #: rows a mouse wheel notch scrolls

def _lit(t, width):
    "`t` padded to `width` on the cursor's background."
    t = t.copy()
    t.pad_right(width - t.cell_len)
    t.stylize(f"on {PALETTE['cursor']}")
    return t

def quit_mark(path):
    "The file a viewer the user quit leaves beside the snapshot at `path`, so the CLI keeps it closed."
    return Path(path).with_suffix('.closed')

class PaneTty(RealTty):
    "`RealTty` whose `read` returns None at the end of input, where `RealTty.read` would spin."
    def read(self, timeout=0.02):
        if not select.select([self.fd], [], [], timeout)[0]: return b''
        return os.read(self.fd, 4096) or None

def _leave(sig, frame): raise SystemExit(0)

class Viewer:
    "The board for the snapshot at `path`, drawn on `tty`: the last good snapshot, the open rows, the cursor and the scroll."
    def __init__(self, path, tty):
        self.path, self.tty, self.parser = Path(path), tty, Parser()
        self.snap = self.good = self.stamp = self.cursor = self.shown = None
        self.open, self.rows, self.top, self.err, self.done, self.size = set(), [], 0, '', False, tty.size

    def reload(self):
        "Read the snapshot again when its file changed; a missing or bad one keeps the last."
        try: st = self.path.stat()
        except OSError: return
        if (stamp := (st.st_ino, st.st_mtime_ns, st.st_size)) == self.stamp: return
        self.stamp = stamp
        if (s := read_snapshot(self.path)) is not None: self.snap, self.err = s, ''

    def toggle(self, key): self.open ^= {key}

    def move(self, key):
        "Put the cursor on `key` and scroll just enough to show it."
        self.cursor, h = key, self.size[1]
        y = next(i for i, (k, _) in enumerate(self.rows) if k == key)
        self.top = y if y < self.top else y - h + 1 if y >= self.top + h else self.top

    def on(self, ev):
        "Act on one input event: a click toggles the row under it, the wheel scrolls, keys move, toggle, scroll or quit."
        keys, h = [k for k, _ in self.rows if k], self.size[1]
        if isinstance(ev, Mouse):
            y = ev.y + self.top
            if ev.press and ev.btn in (64, 65): self.top += WHEEL if ev.btn == 65 else -WHEEL
            elif ev.press and ev.btn == 0 and 0 <= y < len(self.rows) and (k := self.rows[y][0]): self.cursor = k; self.toggle(k)
            return
        if not isinstance(ev, Key): return
        n, i = ev.name, keys.index(self.cursor) if self.cursor in keys else -1
        if n in ('q', 'ctrl+c'): self.done = True
        elif n in ('pageup', 'pagedown'): self.top += (h - 1) * (1 if n == 'pagedown' else -1)
        elif n in ('g', 'G'):
            if keys: self.cursor = keys[0 if n == 'g' else -1]
            self.top = 0 if n == 'g' else len(self.rows)
        elif n in ('j', 'down'):
            if i + 1 < len(keys): self.move(keys[i + 1])
            else: self.top += 1
        elif n in ('k', 'up'):
            if i > 0: self.move(keys[i - 1])
            else: self.top -= 1
        elif n in ('enter', ' ') and i >= 0: self.toggle(self.cursor)

    def paint(self, now=None):
        "Draw the board at the terminal's size, writing only the lines that changed; a new size redraws all."
        w, h = self.tty.size
        if (w, h) != self.size: self.size, self.shown = (w, h), None
        self.rows = board(self.snap, w, now, self.open, self.err)
        self.top = max(0, min(self.top, len(self.rows) - h))
        texts = [_lit(t, w) if k and k == self.cursor else t for k, t in self.rows[self.top:self.top + h]]
        con = Console(file=io.StringIO(), width=w, force_terminal=True, color_system='truecolor', highlight=False)
        with con.capture() as cap: con.print(Group(*texts))
        lines, old = cap.get().split('\n')[:-1], self.shown
        out = ['\x1b[2J'] if old is None else []
        old = old or []
        for y in range(max(len(lines), len(old))):
            new = lines[y] if y < len(lines) else ''
            if new != (old[y] if y < len(old) else ''): out.append(f'\x1b[{y + 1};1H\x1b[K{new}')
        if out: self.tty.write(''.join(out))
        self.shown = lines

    def tick(self, every=0, now=None):
        "Read input for up to `every` seconds, act on it, reload and repaint. An error becomes a row over the last good snapshot."
        try:
            if (data := self.tty.read(every)) is None:
                self.done = True
                return
            for ev in self.parser.feed(data) if data else self.parser.flush(): self.on(ev)
            self.reload()
            self.paint(now)
            self.good = self.snap
        except Exception as e: self.snap, self.err = self.good, _1(f'{type(e).__name__}: {e}', 200)

    def run(self, every=0.2):
        "Draw until q, ctrl+c, a hangup, SIGTERM or the end of input, leave the `quit_mark`, and give the terminal back."
        was = {s: signal.signal(s, _leave) for s in (signal.SIGHUP, signal.SIGTERM)}
        self.tty.write(ENTER)
        try:
            try:
                while not self.done: self.tick(every)
            except (KeyboardInterrupt, SystemExit): pass
            quit_mark(self.path).touch()
        finally:
            for s, h in was.items(): signal.signal(s, h)
            self.tty.write(LEAVE)
            self.tty.restore()

A click on the running sub-agent opens it and puts the cursor there. A tick with nothing new writes nothing. Then `j` moves down, space opens `run_bbb`, and `q` quits.

In [ ]:
class Tty:
    "A terminal for tests: scripted input chunks, a size, and every write kept."
    def __init__(self, *input, size=(40, 30)): self.input, self.size, self.writes = list(input), size, []
    def read(self, timeout=0): return self.input.pop(0).encode() if self.input else b''
    def write(self, s): self.writes.append(s)
    def restore(self): pass
with tempfile.TemporaryDirectory() as d:
    (p := Path(d)/'now.json').write_text(json.dumps(busy))
    v = Viewer(p, Tty()); v.tick(now=now)
    y = next(i for i, (k, _) in enumerate(v.rows) if k == ('sub', 'run_aaa'))
    v.tty.input.append(f'\x1b[<0;3;{y + 1}M'); v.tick(now=now)
    test_eq((v.open, v.cursor), ({('sub', 'run_aaa')}, ('sub', 'run_aaa')))
    n = len(v.tty.writes); v.tick(now=now)
    test_eq(len(v.tty.writes), n)
    v.tty.input += ['j', ' ', 'q']
    for _ in range(3): v.tick(now=now)
    test_eq((v.cursor, v.open, v.done), (('sub', 'run_bbb'), {('sub', 'run_aaa'), ('sub', 'run_bbb')}, True))

`ramabana-pane PATH` runs the viewer on the terminal it was started in.

In [ ]:
#| export
@call_parse
def main(path:str,        # the snapshot file the CLI writes
         every:float=0.2, # seconds to wait for input between frames
        ):
    "Show the `now` snapshot at `path` as a board until q or ctrl+c."
    Viewer(path, PaneTty()).run(every)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()